<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 La Biblioteca con Copias en Varias Sedes: HDFS, Spark y HBase 📚
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 01
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/01%20-%20Procesamiento%20y%20Almacenamiento%20Distribuido/Para%20Dummies/02_Frameworks_Distribuidos_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 02 del Módulo 01 (HDFS, Spark y HBase)](../02_Frameworks_Distribuidos_HDFS_Spark_HBase.ipynb). Imagina una **red de bibliotecas** con **cuatro sedes** (Centro, Norte, Sur y Oriente). Tres ideas, tres herramientas:

| En la red de bibliotecas | En el mundo de los datos |
|---|---|
| Un libro enorme se **parte en tomos** y **cada tomo se guarda en tres sedes**; en la recepción hay un **fichero (catálogo)** que dice dónde está cada copia. Si una sede cierra, **se saca fotocopia** en otra | **HDFS**: guarda archivos gigantes en **bloques replicados**; el **NameNode** es el catálogo |
| Un equipo de bibliotecarios recibe una **receta por escrito** («pasa a limpio, filtra, suma») y **no hace nada hasta que alguien pide el resultado**; si repite la consulta, puede **apuntar el resultado en un cuaderno** para no rehacerlo | **Spark**: plan perezoso (DAG), se ejecuta al **pedir** el resultado; `cache()` guarda lo ya calculado |
| Un **archivo de fichas ordenadas por código**: buscar «de LIB-0100 a LIB-0120» es rápido porque están **en orden**, y cada ficha guarda sus **últimos préstamos** | **HBase**: filas **ordenadas por clave**, con **versiones** y escaneo por rango |

Al terminar podrás explicar:
1. Por qué guardar **3 copias** de cada tomo permite que **una sede cierre sin perder nada** y qué hace el catálogo para recuperarse.
2. Qué significa que Spark sea **perezoso** y para qué sirve **apuntar resultados** (`cache`).
3. Por qué un **archivo ordenado por clave** hace rapidísima una búsqueda **por rango**.

> 🔗 Si luego quieres ver el **mini-HDFS con racks, el mini-RDD, el mini-HBase con regiones** y PySpark real: [cuaderno estándar](../02_Frameworks_Distribuidos_HDFS_Spark_HBase.ipynb). Antes de esto: [contar los votos en varias mesas](01_MapReduce_Dummies.ipynb).

---
## 1. Tomos con Tres Copias y un Fichero Central 🗄️

La biblioteca guarda **4 tomos** de una enciclopedia. Cada tomo se guarda en **3 de las 4 sedes** (así, ninguna sede tiene todo, pero cada tomo está **en tres lugares**). El **catálogo** anota **en qué sedes está cada tomo**. Veamos qué pasa cuando **cierra la sede Norte**.

In [ ]:
sedes = ["Centro", "Norte", "Sur", "Oriente"]
tomos = ["Tomo 1", "Tomo 2", "Tomo 3", "Tomo 4"]
COPIAS = 3

# El catalogo (NameNode): para cada tomo, en que sedes esta
catalogo = {t: {sedes[(i + k) % 4] for k in range(COPIAS)} for i, t in enumerate(tomos)}
assert all(len(s) == COPIAS for s in catalogo.values())                           # cada tomo en 3 sedes distintas
print("Catálogo inicial:")
for t, s in catalogo.items():
    print(f"  {t}: {sorted(s)}")


def cerrar_sede(sede, catalogo):
    """Cierra una sede: el catalogo (por 'heartbeats' perdidos) la borra de todas las listas."""
    return {t: s - {sede} for t, s in catalogo.items()}


def sacar_fotocopias(catalogo, abiertas):
    """El catalogo detecta tomos con menos de 3 copias y manda copiar en otra sede ABIERTA que no lo tenga."""
    nuevo, copias = {t: set(s) for t, s in catalogo.items()}, 0
    for t, s in nuevo.items():
        for sede in abiertas:
            if len(s) >= COPIAS:
                break
            if sede not in s:
                s.add(sede); copias += 1
    return nuevo, copias


# Cierra la sede Norte
despues = cerrar_sede("Norte", catalogo)
con_dos = [t for t, s in despues.items() if len(s) == 2]
print("\nTras cerrar Norte, tomos con solo 2 copias:", con_dos, "| tomos perdidos:", [t for t, s in despues.items() if not s])
assert con_dos == ["Tomo 1", "Tomo 2", "Tomo 4"] and all(len(s) >= 1 for s in despues.values())                          # nada se perdio

abiertas = ["Centro", "Sur", "Oriente"]
reparado, copias = sacar_fotocopias(despues, abiertas)
print(f"El catálogo mandó sacar {copias} fotocopias y ahora cada tomo vuelve a estar en {COPIAS} sedes abiertas.")
assert copias == 3 and all(len(s) == COPIAS and "Norte" not in s for s in reparado.values())                               # nadie queda con menos copias

---
##### 🛠️ Práctica 1: ¿Y si cierran dos sedes a la vez?

Con el `catalogo` inicial (cada tomo en 3 sedes) y la función `cerrar_sede(sede, catalogo)`:

1. Cierra **Norte** y luego **Sur** (aplica `cerrar_sede` dos veces, partiendo de `catalogo`) y guarda el resultado en `dos_cerradas`.
2. Guarda en `perdidos` la lista de tomos **sin ninguna copia** y en `una_copia` la lista de tomos con **exactamente una copia**.
3. Comprueba con `assert` que `perdidos == []` y que `una_copia == ["Tomo 1", "Tomo 2"]`. Comenta: ¿qué pasaría si cerraran **tres** sedes a la vez? ¿Por qué conviene **sacar las fotocopias rápido** después del primer cierre?

In [ ]:
# Escribe tu código aquí

# dos_cerradas = ...
# perdidos = ...
# una_copia = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
dos_cerradas = cerrar_sede("Sur", cerrar_sede("Norte", catalogo))
perdidos = [t for t, s in dos_cerradas.items() if len(s) == 0]
una_copia = [t for t, s in dos_cerradas.items() if len(s) == 1]
print("Sin copias:", perdidos, "| con una sola copia:", una_copia)

assert perdidos == []
assert una_copia == ["Tomo 1", "Tomo 2"]
# Con 3 sedes cerradas a la vez, un tomo cuyas tres copias estaban justo en esas sedes se perderia. Por eso, tras el PRIMER cierre,
# el catalogo saca fotocopias enseguida: asi un segundo cierre no encuentra tomos con una sola copia.
```
</details>

---


---
## 2. Los Bibliotecarios con una Receta (Spark) 📝

Un equipo de bibliotecarios recibe una **receta**: «1) de la lista de préstamos, quédate con los del mes; 2) pasa los montos a pesos; 3) suma por sección». **Escribir la receta no gasta esfuerzo**: **nadie hace nada** hasta que alguien **pide el resultado**. Eso permite que el equipo **vea toda la receta** y la organice mejor.

Y hay otro truco: si te piden **cinco informes** que empiezan con los mismos dos pasos caros, puedes **apuntar en un cuaderno** el resultado intermedio (`cache`) y **no rehacerlo** cinco veces.

In [ ]:
class Receta:
    """Una receta perezosa: guarda los pasos y solo los ejecuta cuando alguien 'pide' el resultado."""
    pasos_hechos = 0                                                            # cuantas veces un bibliotecario ejecuto un paso

    def __init__(self, datos, pasos=(), apuntar=False):
        self.datos, self.pasos, self.apuntar, self._cuaderno = datos, tuple(pasos), apuntar, None

    def luego(self, f):
        return Receta(self.datos, self.pasos + (f,), self.apuntar)               # escribir la receta NO hace nada

    def guardar_en_cuaderno(self):
        self.apuntar = True
        return self

    def pedir(self):
        if self.apuntar and self._cuaderno is not None:
            return self._cuaderno                                               # ya estaba apuntado: no se rehace
        resultado = self.datos
        for paso in self.pasos:
            resultado = [paso(x) for x in resultado]
            Receta.pasos_hechos += len(resultado)
        if self.apuntar:
            self._cuaderno = resultado
        return resultado


def a_pesos(dolares):
    return dolares * 4000                                                       # tasa inventada


def con_iva(pesos):
    return round(pesos * 1.19)


prestamos = [1, 2, 3, 4, 5]

Receta.pasos_hechos = 0
receta = Receta(prestamos).luego(a_pesos).luego(con_iva)
assert Receta.pasos_hechos == 0                                                 # escribir la receta no gasto esfuerzo
resultado = receta.pedir()
assert Receta.pasos_hechos == 10 and resultado == [4760, 9520, 14280, 19040, 23800]
print(f"Receta escrita: 0 pasos. Al pedir el resultado: {Receta.pasos_hechos} pasos (2 pasos x 5 datos) -> {resultado}")

# Cinco informes sobre los mismos dos pasos caros: sin cuaderno se repite todo; con cuaderno se hace una vez
Receta.pasos_hechos = 0
sin_cuaderno = Receta(prestamos).luego(a_pesos).luego(con_iva)
for _ in range(5): sin_cuaderno.pedir()
repetidos = Receta.pasos_hechos
Receta.pasos_hechos = 0
con_cuaderno = Receta(prestamos).luego(a_pesos).luego(con_iva).guardar_en_cuaderno()
for _ in range(5): con_cuaderno.pedir()
una_vez = Receta.pasos_hechos
print(f"Cinco informes: sin cuaderno = {repetidos} pasos | con cuaderno (cache) = {una_vez} pasos")
assert repetidos == 50 and una_vez == 10

---
**Lección:** escribir la receta **no hizo ningún paso**; los **10 pasos** (2 por cada uno de los 5 datos) se hicieron **solo al pedir el resultado**. Y con **cinco informes**, repetir todo cuesta **50 pasos**, mientras que **apuntar el resultado intermedio** cuesta **solo 10**. En Spark real el «cuaderno» es la **memoria de los ejecutores** (`cache`), y la «receta» es el **plan (DAG)**: además, si un bibliotecario se enferma, **otro rehace sus pasos desde la receta** (por eso Spark no necesita guardar 3 copias de todo como HDFS).

---

## 3. El Archivo de Fichas Ordenadas (HBase) 🗃️

El archivo de la biblioteca guarda **fichas ordenadas por código** (`LIB-0001`, `LIB-0002`...). Como están **en orden**, para encontrar las fichas **de LIB-0100 a LIB-0120** no hace falta revisar todas: se **salta directo** a la LIB-0100 y se **lee hacia adelante** hasta pasar la 0120. Además cada ficha guarda **varias versiones** de un dato (por ejemplo, **los últimos préstamos**), de la más nueva a la más vieja.

In [ ]:
import bisect

codigos = [f"LIB-{n:04d}" for n in range(1, 501)]                              # 500 fichas ya ordenadas
assert codigos == sorted(codigos)
prestamos_ficha = {"LIB-0105": [("2026-03-20", "Marta"), ("2026-02-11", "Luis"), ("2026-01-05", "Ana")]}   # versiones: la mas nueva primero


def fichas_entre(desde, hasta):
    """Busqueda por rango: salta directo al inicio (busqueda binaria) y lee hacia adelante hasta pasar el final."""
    i = bisect.bisect_left(codigos, desde)
    out, revisadas = [], 0
    while i < len(codigos) and codigos[i] <= hasta:
        out.append(codigos[i]); i += 1; revisadas += 1
    return out, revisadas


fichas, leidas = fichas_entre("LIB-0100", "LIB-0120")
print(f"Fichas de LIB-0100 a LIB-0120: {len(fichas)} fichas (se leyeron {leidas} de {len(codigos)}).")
assert len(fichas) == 21 and fichas[0] == "LIB-0100" and fichas[-1] == "LIB-0120" and leidas == 21 < len(codigos)

ultimos = prestamos_ficha["LIB-0105"][:2]                                       # las 2 versiones mas nuevas
print("Últimos dos préstamos de LIB-0105:", ultimos)
assert ultimos == [("2026-03-20", "Marta"), ("2026-02-11", "Luis")]

# Contraste: buscar por un dato que NO es el codigo (por ejemplo, 'quien presto algo') obliga a revisar todo el archivo
quien = [c for c in codigos if c in prestamos_ficha and prestamos_ficha[c][0][1] == "Marta"]
assert quien == ["LIB-0105"]
print("Buscar por la persona (no por el código) obligaría a mirar las 500 fichas, a menos que armemos otro archivo ordenado por persona.")

---
**Lección:** para **21 fichas** en medio de **500** solo se **leyeron 21**: como el archivo está **ordenado**, se salta directo al inicio. Buscar **por otro dato** (quién prestó) sí obligaría a **revisar todo** (o a mantener **otro archivo ordenado** por ese dato). Por eso en HBase **la clave de fila lo es todo**: se diseña **pensando en cómo se va a consultar**.

---

## Resumen en una frase 🎯

> **HDFS guarda cada tomo en tres sedes con un catálogo que repara las pérdidas, Spark es un equipo que no mueve un dedo hasta que le piden el resultado (y apunta lo repetido), y HBase es un archivo ordenado por código donde los rangos se encuentran saltando directo al inicio.**

### Lo que aprendiste hoy:
- ✅ **Tres copias y un catálogo:** al cerrar Norte, 3 tomos quedaron con 2 copias, **ninguno se perdió** y el catálogo mandó **3 fotocopias**; con dos sedes cerradas, 2 tomos quedaron con una sola copia (por eso se repara rápido).
- ✅ **Receta perezosa y cuaderno:** 0 pasos al escribir, 10 al pedir; cinco informes cuestan **50 pasos sin cuaderno** y **10 con él** (`cache`).
- ✅ **Fichas ordenadas:** el rango LIB-0100 a LIB-0120 se lee con **21 fichas** de 500; buscar por otro dato exige otro orden (o revisar todo).

> 🎓 **Siguiente paso:** [el cuaderno estándar](../02_Frameworks_Distribuidos_HDFS_Spark_HBase.ipynb) construye un mini-HDFS con racks y re-replicación, un mini-RDD con etapas y una tabla estilo HBase con regiones, y los contrasta con PySpark. Con esto cierras el módulo; sigue el **Módulo 02: *Data Warehousing* y *Data Lake***.

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿qué hace el **catálogo** cuando una sede cierra y por qué es importante que lo haga **rápido**?
2. Si Spark es «perezoso», ¿en qué momento se hace el trabajo? ¿Qué ventaja tiene que el equipo **vea toda la receta** antes de empezar?
3. Tienes un archivo de fichas ordenado por **código**. ¿Qué consultas son rápidas y cuáles lentas? ¿Qué harías para que **buscar por apellido** también sea rápido?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Big Data (Edición Didáctica Para Dummies)</i>
  </p>
</div>